In [1]:
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# --- 1. Load SMS Spam Data ---
print("Loading SMS Spam data...")
# Note: If your file is a .tsv, change 'sep=','' to 'sep='\t''. 
# If it has no headers, use: pd.read_csv(path, sep=',', names=['label', 'text'])
sms_path = "../data/training/sms_spam_uci.csv"
try:
    sms_df = pd.read_csv(sms_path, encoding='latin-1', on_bad_lines='skip')
    # Clean up UCI dataset format (usually has v1, v2 columns)
    if 'v1' in sms_df.columns and 'v2' in sms_df.columns:
        sms_df = sms_df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
    else:
        # If it's the tab-separated version without headers
        sms_df = pd.read_csv(sms_path, sep='\t', names=['label', 'text'], encoding='latin-1', on_bad_lines='skip')
    
    # Convert ham/spam to 0/1
    sms_df['label'] = sms_df['label'].map({'ham': 0, 'spam': 1})
except Exception as e:
    print(f"Error loading SMS data: {e}")
    sms_df = pd.DataFrame(columns=['label', 'text'])


# --- 2. Load Phishing Validation Data ---
print("Loading Phishing Validation data...")
val_path = "../data/training/phishing_validation.xlsx"
try:
    val_df = pd.read_excel(val_path)
    # Rename columns to match our standard
    val_df = val_df.rename(columns={'Email Text': 'text', 'Email Type': 'label'})
    # Convert Safe Email/Phishing Email to 0/1
    val_df['label'] = val_df['label'].map({'Safe Email': 0, 'Phishing Email': 1})
    val_df = val_df[['label', 'text']]
except Exception as e:
    print(f"Error loading Validation data: {e}")
    val_df = pd.DataFrame(columns=['label', 'text'])


# --- 3. Load Multiclass NLP Data ---
print("Loading Multiclass NLP data...")
multi_path = "../data/training/phishing_nlp_dataset.xlsx"
try:
    multi_df = pd.read_excel(multi_path)
    multi_df = multi_df.rename(columns={'Corpus': 'text', 'Label': 'label'})
    # Convert everything that isn't NOT-Malicious to 1 (Malicious)
    multi_df['label'] = multi_df['label'].apply(lambda x: 0 if str(x).strip().upper() == 'NOT-MALICIOUS' else 1)
    multi_df = multi_df[['label', 'text']]
except Exception as e:
    print(f"Error loading Multiclass data: {e}")
    multi_df = pd.DataFrame(columns=['label', 'text'])


# --- 4. Combine all datasets ---
print("Combining datasets...")
combined_df = pd.concat([sms_df, val_df, multi_df], ignore_index=True)

# Drop any rows where text or label is missing
combined_df = combined_df.dropna()

print(f"Total combined rows: {len(combined_df)}")
print(f"Label distribution:\n{combined_df['label'].value_counts()}")

# --- 5. Train the Model ---
print("Training the model...")
X = combined_df['text'].astype(str)
y = combined_df['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

# Convert text to numbers (TF-IDF)
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Logistic Regression
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate
y_pred = model.predict(X_test_vec)
print("\n--- Evaluation on Test Set ---")
print(classification_report(y_test, y_pred, target_names=["Safe (0)", "Suspicious (1)"]))

# --- 6. Save the Model ---
print("Saving model and vectorizer...")
os.makedirs("../backend/models", exist_ok=True)
joblib.dump(model, "../backend/models/phishing_model.pkl")
joblib.dump(vectorizer, "../backend/models/tfidf_vectorizer.pkl")
print("Successfully saved to backend/models/")

Loading SMS Spam data...
Loading Phishing Validation data...
Error loading Validation data: [Errno 2] No such file or directory: '../data/training/phishing_validation.xlsx'
Loading Multiclass NLP data...
Error loading Multiclass data: 'label'
Combining datasets...
Total combined rows: 5572
Label distribution:
label
0    4825
1     747
Name: count, dtype: int64
Training the model...

--- Evaluation on Test Set ---
                precision    recall  f1-score   support

      Safe (0)       0.98      0.99      0.98       724
Suspicious (1)       0.91      0.89      0.90       112

      accuracy                           0.97       836
     macro avg       0.95      0.94      0.94       836
  weighted avg       0.97      0.97      0.97       836

Saving model and vectorizer...
Successfully saved to backend/models/


In [4]:
# --- 2. Load Phishing Validation Data ---
print("Loading Phishing Validation data...")
try:
    # Changed to read_csv since it is a CSV file
    val_df = pd.read_csv("../data/training/phishing_validation.csv")
    
    # Rename columns to match our standard
    val_df = val_df.rename(columns={'Email Text': 'text', 'Email Type': 'label'})
    # Convert to 0 and 1
    val_df['label'] = val_df['label'].str.strip().map({'Safe Email': 0, 'Phishing Email': 1})
    val_df = val_df[['label', 'text']].dropna()
    
    print(f"Successfully loaded {len(val_df)} rows from Validation data.")
except Exception as e:
    print(f"Error loading Validation data: {e}")
    val_df = pd.DataFrame(columns=['label', 'text'])

Loading Phishing Validation data...
Successfully loaded 2000 rows from Validation data.


In [7]:
# --- 3. Load Multiclass NLP Data ---
print("Loading Multiclass NLP data...")
try:
    # Using read_excel since it is an .xlsx file
    multi_df = pd.read_excel("../data/training/phishing_nlp.xlsx")
    
    # Since text and label are in one column, we split them using this function
    def parse_multi_row(row):
        text_val = str(row.iloc[0]).strip()
        
        malicious_labels = ['Phishing', 'Malware', 'Scareware', 'Baiting', 'Pretexting']
        safe_labels = ['NOT-Malicious']
        
        for label in malicious_labels:
            if text_val.endswith(label):
                clean_text = text_val[:-len(label)].strip()
                return pd.Series([1, clean_text])
                
        for label in safe_labels:
            if text_val.endswith(label):
                clean_text = text_val[:-len(label)].strip()
                return pd.Series([0, clean_text])
                
        return pd.Series([0, text_val])

    multi_df[['label', 'text']] = multi_df.apply(parse_multi_row, axis=1)
    multi_df = multi_df[['label', 'text']].dropna()
    
    print(f"Successfully loaded {len(multi_df)} rows from Multiclass data.")
except Exception as e:
    print(f"Error loading Multiclass data: {e}")
    multi_df = pd.DataFrame(columns=['label', 'text'])

Loading Multiclass NLP data...
Successfully loaded 624 rows from Multiclass data.


In [8]:
# --- 4. Combine all datasets ---
print("Combining datasets...")
combined_df = pd.concat([sms_df, val_df, multi_df], ignore_index=True)
combined_df['label'] = combined_df['label'].astype(int)

print(f"Total combined rows: {len(combined_df)}")
print(f"Label distribution:\n{combined_df['label'].value_counts()}")

# --- 5. Train the Model ---
print("\nTraining the model...")
X = combined_df['text'].astype(str)
y = combined_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_vec, y_train)

# --- 6. Evaluate ---
y_pred = model.predict(X_test_vec)
print("\n--- Evaluation on Test Set ---")
print(classification_report(y_test, y_pred, target_names=["Safe (0)", "Suspicious (1)"]))

# --- 7. Save the Model ---
print("Saving model and vectorizer...")
joblib.dump(model, "../backend/models/phishing_model.pkl")
joblib.dump(vectorizer, "../backend/models/tfidf_vectorizer.pkl")
print("Successfully saved to backend/models/")

Combining datasets...
Total combined rows: 8196
Label distribution:
label
0    5999
1    2197
Name: count, dtype: int64

Training the model...

--- Evaluation on Test Set ---
                precision    recall  f1-score   support

      Safe (0)       0.99      0.99      0.99       900
Suspicious (1)       0.98      0.96      0.97       330

      accuracy                           0.98      1230
     macro avg       0.98      0.98      0.98      1230
  weighted avg       0.98      0.98      0.98      1230

Saving model and vectorizer...
Successfully saved to backend/models/
